In [0]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("Spark DataFrames").getOrCreate()
from pyspark.sql.functions import *
data = [
(101,"Arjun Reddy","Hyderabad","Cardiology",5000,1),
(102,"Sneha Kapoor","Delhi","Orthopedics",3000,2),
(103,"Rahul Sharma","Mumbai","Dermatology",1500,1),
(104,"Priya Nair","Bangalore","Cardiology",5000,2),
(105,"Vikram Singh","Chennai","Neurology",7000,1),
(106,"Ananya Das","Kolkata","Orthopedics",3000,3),
(107,"Karan Patel","Ahmedabad","Cardiology",5000,1),
(108,"Meera Iyer","Bangalore","Dermatology",1500,2)
]
columns = [
"visit_id",
"patient_name",
"city",
"department",
"consultation_fee",
"tests_count"
]

df = spark.createDataFrame(data, columns)
df.show()

+--------+------------+---------+-----------+----------------+-----------+
|visit_id|patient_name|     city| department|consultation_fee|tests_count|
+--------+------------+---------+-----------+----------------+-----------+
|     101| Arjun Reddy|Hyderabad| Cardiology|            5000|          1|
|     102|Sneha Kapoor|    Delhi|Orthopedics|            3000|          2|
|     103|Rahul Sharma|   Mumbai|Dermatology|            1500|          1|
|     104|  Priya Nair|Bangalore| Cardiology|            5000|          2|
|     105|Vikram Singh|  Chennai|  Neurology|            7000|          1|
|     106|  Ananya Das|  Kolkata|Orthopedics|            3000|          3|
|     107| Karan Patel|Ahmedabad| Cardiology|            5000|          1|
|     108|  Meera Iyer|Bangalore|Dermatology|            1500|          2|
+--------+------------+---------+-----------+----------------+-----------+



In [0]:
df = df.withColumn("total_bill", col("consultation_fee") + (col("tests_count") * 500))

In [0]:
high_value_df = df.filter(col("total_bill") > 6000)
high_value_df.show()

+--------+------------+-------+----------+----------------+-----------+----------+
|visit_id|patient_name|   city|department|consultation_fee|tests_count|total_bill|
+--------+------------+-------+----------+----------------+-----------+----------+
|     105|Vikram Singh|Chennai| Neurology|            7000|          1|      7500|
+--------+------------+-------+----------+----------------+-----------+----------+



In [0]:
dept_agg = df.groupBy("department") \
    .agg(
        sum("total_bill").alias("total_revenue"),
        avg("total_bill").alias("avg_revenue")
    )

In [0]:
dept_agg.orderBy(desc("total_revenue")).show()

+-----------+-------------+-----------------+
| department|total_revenue|      avg_revenue|
+-----------+-------------+-----------------+
| Cardiology|        17000|5666.666666666667|
|Orthopedics|         8500|           4250.0|
|  Neurology|         7500|           7500.0|
|Dermatology|         4500|           2250.0|
+-----------+-------------+-----------------+



In [0]:
df.createOrReplaceTempView("hospital_db")

In [0]:
%sql
select * from hospital_db where department = "Cardiology"

visit_id,patient_name,city,department,consultation_fee,tests_count,total_bill
101,Arjun Reddy,Hyderabad,Cardiology,5000,1,5500
104,Priya Nair,Bangalore,Cardiology,5000,2,6000
107,Karan Patel,Ahmedabad,Cardiology,5000,1,5500


In [0]:
%sql
select city, sum(total_bill) from hospital_db group by city

city,sum(total_bill)
Hyderabad,5500
Delhi,4000
Mumbai,2000
Bangalore,8500
Chennai,7500
Kolkata,4500
Ahmedabad,5500


In [0]:
%sql
SELECT patient_name, total_bill
FROM hospital_db
ORDER BY total_bill DESC
LIMIT 3;

patient_name,total_bill
Vikram Singh,7500
Priya Nair,6000
Arjun Reddy,5500


In [0]:
%sql
SELECT department, COUNT(*) AS patient_count
FROM hospital_db
GROUP BY department;

department,patient_count
Cardiology,3
Orthopedics,2
Dermatology,2
Neurology,1


In [0]:
%sql
SELECT department, COUNT(*) AS patient_count
FROM hospital_db
GROUP BY department;

department,patient_count
Cardiology,3
Orthopedics,2
Dermatology,2
Neurology,1


In [0]:
df.write.format("delta").mode("overwrite").save("/mnt/delta/hospital")

delta_df = spark.read.format("delta").load("/mnt/delta/hospital")

---------------------------------------------------------------------------
UnsupportedOperationException             Traceback (most recent call last)
File <command-6318581880770624>, line 1
----> 1 df.write.format("delta").mode("overwrite").save("/mnt/delta/hospital")
      3 delta_df = spark.read.format("delta").load("/mnt/delta/hospital")

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/readwriter.py:703, in DataFrameWriter.save(self, path, format, mode, partitionBy, **options)
    701     self.format(format)
    702 self._write.path = path
--> 703 _, _, ei = self._spark.client.execute_command(
    704     self._write.command(self._spark.client), self._write.observations
    705 )
    706 self._callback(ei)

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/client/core.py:1538, in SparkConnectClient.execute_command(self, command, observations, extra_request_metadata)
   1536     req.user_context.user_id = self._user_id
   1537 self._s

In [0]:
df.write.format("delta") \
  .mode("overwrite") \
  .saveAsTable("hospital_table")

In [0]:
delta_df = spark.read.table("hospital_table")

In [0]:
new_data = [(109,"Ravi Kumar","Pune","Cardiology",5000,2)]
new_df = spark.createDataFrame(new_data, columns)

new_df = new_df.withColumn("total_bill", col("consultation_fee") + col("tests_count")*500)

new_df.write.format("delta").mode("append").saveAsTable("hospital_table")

In [0]:
from delta.tables import DeltaTable
delta_table = DeltaTable.forName(spark, "hospital_table")

# UPDATE
delta_table.update(
    condition="visit_id = 101",
    set={"consultation_fee": "5500"}
)

# DELETE
delta_table.delete("visit_id = 108")

# UPSERT
updates = [(101,"Arjun Reddy","Hyderabad","Cardiology",6000,2)]
updates_df = spark.createDataFrame(updates, columns)

updates_df = updates_df.withColumn("total_bill", col("consultation_fee") + col("tests_count")*500)

delta_table.alias("t").merge(
    updates_df.alias("s"),
    "t.visit_id = s.visit_id"
).whenMatchedUpdateAll() \
 .whenNotMatchedInsertAll() \
 .execute()

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

In [0]:
%sql

DESCRIBE HISTORY hospital_table;

SELECT * FROM hospital_table VERSION AS OF 0;


VACUUM hospital_table RETAIN 168 HOURS DRY RUN;

path


In [0]:
df.write.mode("overwrite").parquet("/tmp/hospital_parquet")
CONVERT TO DELTA parquet.`/tmp/hospital_parquet`;

---------------------------------------------------------------------------
UnsupportedOperationException             Traceback (most recent call last)
File <command-6318581880770632>, line 1
----> 1 df.write.mode("overwrite").parquet("/tmp/hospital_parquet")

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/readwriter.py:779, in DataFrameWriter.parquet(self, path, mode, partitionBy, compression)
    777     self.partitionBy(partitionBy)
    778 self._set_opts(compression=compression)
--> 779 self.format("parquet").save(path)

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/readwriter.py:703, in DataFrameWriter.save(self, path, format, mode, partitionBy, **options)
    701     self.format(format)
    702 self._write.path = path
--> 703 _, _, ei = self._spark.client.execute_command(
    704     self._write.command(self._spark.client), self._write.observations
    705 )
    706 self._callback(ei)

File /databricks/python/lib/python3.12/sit

In [0]:
updates = [
(101,"Arjun Reddy","Hyderabad","Cardiology",6500,2),
(110,"New Patient","Delhi","Neurology",8000,1)
]

updates_df = spark.createDataFrame(updates, columns)

updates_df = updates_df.withColumn("total_bill", col("consultation_fee") + col("tests_count")*500)

target = DeltaTable.forName(spark, "hospital_table")

target.alias("t").merge(
    updates_df.alias("s"),
    "t.visit_id = s.visit_id"
).whenMatchedUpdateAll() \
 .whenNotMatchedInsertAll() \
 .execute()

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

In [0]:
%sql
CREATE CATALOG if not exists hospital_catalg;
CREATE SCHEMA if not exists hospital_catalg.healthcare;

CREATE TABLE if not exists hospital_catalg.healthcare.patient_data
USING DELTA
AS SELECT * FROM hospital_table;

num_affected_rows,num_inserted_rows


In [0]:
%sql
-- Discover tables
SHOW TABLES IN hospital_catalg.healthcare;

-- Derived table
CREATE TABLE IF NOT EXISTS hospital_catalg.healthcare.high_value_patients AS
SELECT * FROM hospital_catalg.healthcare.patient_data
WHERE total_bill > 6000;

-- Access control


num_affected_rows,num_inserted_rows
